In [ ]:
!pip install spacy

In [ ]:
import pandas as pd
import spacy
from spacy import displacy
from spacy.tokens import DocBin
import json
from datetime import datetime
from tqdm import tqdm
import re

collective_dict = {'TRAINING_DATA': []}

def structure_training_data(text, kw_list):
    results = []
    entities = []

    # Search for instances of keywords within the text (ignoring letter case)
    for kw in tqdm(kw_list):
        search = re.finditer(kw, text, flags=re.IGNORECASE)

        # Store the start/end character positions
        all_instances = [[m.start(), m.end()] for m in search]

        # If the collable_iterator found matches, create an entities' List
        if len(all_instances) > 0:
            for i in all_instances:
                start = i[0]
                end = i[1]
                entities.append((start, end, "SERVICE"))

        # Alert when no matches are found given the user inputs
        else:
            print("No pattern matches found. Keyword:", kw)

    # Add any found entities into a JSON format within collective dict
    if len(entities) > 0:
        results = [text, {"entities": entities}]
        collective_dict["TRAINING_DATA"].append(results)
    return

# Example usage:
text = "Some sample text mentioning services like laundry, cleaning, and catering."
kw_list = ["laundry", "cleaning", "catering"]
structure_training_data(text, kw_list)
print(collective_dict)


100%|██████████| 3/3 [00:00<00:00, 3266.59it/s]

{'TRAINING_DATA': [['Some sample text mentioning services like laundry, cleaning, and catering.', {'entities': [(42, 49, 'SERVICE'), (51, 59, 'SERVICE'), (65, 73, 'SERVICE')]}]]}


In [ ]:
text1 = "BigTime Care has a broad array of service offerings for Philadelphia-area clientele.\
For 50 years, we have specialized in landscaping and lawn mowing.\
We also provide seasonal snow removal services for local commercial and residential properties.\
Call any time to schedule a consultation!"

text2 = "Scrub-O Cleaning connects independent professionals with customers. \
We offer the full range of customizable cleaning services that you may need now and in \
the future, and our team is ready to begin working for you today! We offer quality maid \
services and housekeeping across the San Francisco Bay Area."

text3 = "Locally owned and operated, Trust Roofing has the best roofing services in \
Philadelphia and the surrounding areas. Whatever the season, you can count on us to provide \
you with the best possible roof repair.\ We will work with any given roof replacement material, \
including asphalt shingles and metal roofs. Siding replacement services are also available."

text4 = "Based in Pittsburgh PA, Tammy's Branch Cuts is a family owned and managed small \
businesses founded in 1994. We specialize in full-service landscape design, including \
tree removal, lawn care to protect your existing plants, and comprehensive hardscaping for \
patios, walkways, and outdoor living spaces. Contact us today!"

# TRAINING
structure_training_data(text1, ['landscaping', 'lawn mowing', 'snow removal'])
structure_training_data(text2, ['cleaning services', 'maid services', 'housekeeping'])
structure_training_data(text3, ['roofing', 'roof repair', 'siding replacement'])
structure_training_data(text4, ['landscape design', 'tree removal', 'lawn care','hardscaping'])

100%|██████████| 4/4 [00:00<00:00, 6234.57it/s]


In [ ]:
#define our training data to TRAIN_DATA
TRAIN_DATA  = collective_dict['TRAINING_DATA']

#create a blank model
nlp = spacy.blank("en")

def create_training(TRAIN_DATA):
    db = DocBin()
    for text, annot in tqdm(TRAIN_DATA):
        doc = nlp.make_doc(text)
        ents = []

        #create span objects
        for start, end, label in annot["entities"]:
            span = doc.char_span(start, end, label=label, alignment_mode="contract")

            #skip if the character indices do not map to a valid span
            if span is None:
                print("Skipping entity.")
            else:
                ents.append(span)
        # handle erroneous entity annotations by removing them
        try:
            doc.ents = ents
        except Exception as e:
            print("Error:", e)
            ents = []

        doc.ents = ents

        #pack Doc objects into DocBin
        db.add(doc)
    return db

TRAIN_DATA_DOC = create_training(TRAIN_DATA)

#Export results (here I add it to a TRAIN DATA folder within the directory)
TRAIN_DATA_DOC.to_disk("./train.spacy")

100%|██████████| 5/5 [00:00<00:00, 155.75it/s]


In [ ]:
# !python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_lg

In [ ]:
!python -m spacy init fill-config base_config.cfg config.cfg

✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
!python -m spacy train config.cfg --output ./output

ℹ Saving to output directory: output
ℹ Using CPU

=========================== Initializing pipeline ===========================
Traceback (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/local/lib/python3.10/dist-packages/spacy/__main__.py", line 4, in <module>
    setup_cli()
  File "/usr/local/lib/python3.10/dist-packages/spacy/cli/_util.py", line 87, in setup_cli
    command(prog_name=COMMAND)
  File "/usr/local/lib/python3.10/dist-packages/click/core.py", line 1157, in __call__
    return self.main(*args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/typer/core.py", line 778, in main
    return _main(
  File "/usr/local/lib/python3.10/dist-packages/typer/core.py", line 216, in _main
    rv = self.invoke(ctx)
  File "/usr/local/lib/python3.10/dist-packages/click/core.py", line 

In [ ]:
# !pip install -U spacy
# !python -m spacy download en_core_web_lg
# import spacy

# nlp_output = spacy.load("en_core_web_lg")

In [ ]:
!python -m spacy train config.cfg --output ./ --paths.train ./train.spacy --paths.dev ./train.spacy

ℹ Saving to output directory: .
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     20.83    0.00    0.00    0.00    0.00
 71     200          2.95    486.15  100.00  100.00  100.00    1.00
169     400          0.03      0.07  100.00  100.00  100.00    1.00
269     600          0.00      0.00  100.00  100.00  100.00    1.00
426     800          0.00      0.00  100.00  100.00  100.00    1.00
626    1000          0.00      0.00  100.00  100.00  100.00    1.00
826    1200          0.00      0.00  100.00  100.00  100.00    1.00
1026    1400          0.00      0.00  100.00  100.00  100.00    1.00
1226    1600          0.00      0.0

In [ ]:
import spacy
model_test = """At Perfection Landscapes LLC, we are committed to protecting the health of trees \
and shrubs in urban and suburban areas. We work with clients to provide expertise in all areas \
of tree care, stump removal, and construction-related tree preservation. Our trained experts \
also have years of experience with insect control. Call us today for a consultation!"""


#Load the trained model
nlp_output = spacy.load("model-best")

#pass our test instance into the trained pipeline
doc = nlp_output(model_test)

#customize the Label colors
colors = {"SERVICE": "linear-gradient(90deg, #E1D436, #F59710)"}
options = {"ents": ["SERVICE"], "colors": colors}

#visualize the identified entities
spacy.displacy.render(doc, style="ent", options= options, jupyter=True)

#print out the identified entities
for ent in doc.ents:
  if ent.label == "SERVICE":
    print(ent.text, ent.label_)

2nd trial

In [1]:
import spacy

In [2]:
nlp = spacy.load('en_core_web_sm')
nlp.pipe_names

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

In [3]:
doc = nlp("Australia wants to force Facebook and Google to pay media companies for news")

In [4]:
for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

Australia 0 9 GPE
Google 38 44 ORG


In [5]:
doc1 = nlp("I do not have money to pay my credit card account")

In [6]:
for ent in doc1.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

In [7]:
train =[
    ("Money transfer from my checking account is not working",{"entities": [(6,13,"ACTIVITY"),(23,39,'PRODUCT')]}),
    ("I want to check balance in my savings account",{"entities": [(6,13,"ACTIVITY"),(30,45,'PRODUCT')]}),
    ("I suspect a fraud in my credit card account",{"entities": [(12,17,"ACTIVITY"),(24,35,'PRODUCT')]}),
    ("I am here for opening a new savings account",{"entities": [(14,21,"ACTIVITY"),(28,43,'PRODUCT')]}),
    ("Your mortgage is in delinquent status",{"entities": [(20,30,"ACTIVITY"),(5,13,'PRODUCT')]}),
    ("Your credit card is in past due status",{"entities": [(23,31,"ACTIVITY"),(5,16,'PRODUCT')]}),
    ("When is the payment due date on my card",{"entities": [(12,19,"ACTIVITY"),(35,39,'PRODUCT')]}),
    ("Can you help me updating payment on my credit card",{"entities": [(22,29,"ACTIVITY"),(36,47,'PRODUCT')]})
    ]

In [8]:
nlp.pipe_names

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

In [9]:
ner = nlp.get_pipe("ner")

In [10]:
for _, annotations in train:
  for ent in annotations.get("entities"):
    ner.add_label(ent[2])

In [11]:
disable_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']

In [12]:
import random
from spacy.training.example import Example
from spacy.util import minibatch, compounding
from pathlib import Path

with nlp.disable_pipes(*disable_pipes):
    optimizer = nlp.resume_training()

    for iteration in range(100):
        random.shuffle(train)
        losses = {}

        batches = minibatch(train, size=compounding(1.0, 4.0, 1.001))
        for batch in batches:
            examples = []
            for text, annotation in batch:
                examples.append(Example.from_dict(nlp.make_doc(text), annotation))
            nlp.update(
                examples,
                drop=0.5,
                losses=losses,
                sgd=optimizer
            )
            print("Losses", losses)


Losses {'ner': 4.202827464124102}
Losses {'ner': 4.202839579507205}
Losses {'ner': 8.202403671257722}
Losses {'ner': 10.161936646704282}
Losses {'ner': 13.478190929777048}


/usr/local/lib/python3.10/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Can you help me updating payment on my credit card" with entities "[(22, 29, 'ACTIVITY'), (36, 47, 'PRODUCT')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Money transfer from my checking account is not wor..." with entities "[(6, 13, 'ACTIVITY'), (23, 39, 'PRODUCT')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "I w

Losses {'ner': 15.47316511364196}
Losses {'ner': 19.233371734577304}
Losses {'ner': 24.612690142643103}
Losses {'ner': 3.9736938963934776}
Losses {'ner': 5.945847934138259}
Losses {'ner': 5.946204127326478}
Losses {'ner': 9.1123536966828}
Losses {'ner': 13.213520644297642}
Losses {'ner': 16.914049881764146}
Losses {'ner': 20.586264916907343}
Losses {'ner': 22.526860111122314}
Losses {'ner': 3.9730782773847864}
Losses {'ner': 8.149211128544096}
Losses {'ner': 11.180366359606335}
Losses {'ner': 14.417631901694033}
Losses {'ner': 19.080933871204973}
Losses {'ner': 22.204112975173114}
Losses {'ner': 22.206277706651736}
Losses {'ner': 24.343663207442656}
Losses {'ner': 6.042010178877717}
Losses {'ner': 8.952863938812698}
Losses {'ner': 11.900937545314264}
Losses {'ner': 13.539118153976554}
Losses {'ner': 20.854450800385486}
Losses {'ner': 24.540332450147936}
Losses {'ner': 27.385327948745875}
Losses {'ner': 27.38731623220464}
Losses {'ner': 1.270607917693415}
Losses {'ner': 4.01375934718831

In [13]:
for text, _ in train:
    doc = nlp(text)
    print('Entities',[(ent.text,ent.label_) for ent in doc.ents])

Entities [('checking account', 'PRODUCT')]
Entities [('fraud', 'ACTIVITY'), ('credit card', 'PRODUCT')]
Entities [('savings account', 'PRODUCT')]
Entities [('payment', 'ACTIVITY'), ('card', 'PRODUCT')]
Entities [('mortgage', 'PRODUCT'), ('delinquent', 'ACTIVITY')]
Entities [('opening', 'ACTIVITY'), ('savings account', 'PRODUCT')]
Entities [('credit card', 'PRODUCT'), ('past due', 'ACTIVITY')]
Entities [('payment', 'ACTIVITY'), ('credit card', 'PRODUCT')]


In [14]:
from spacy import displacy

doc = nlp("what is the process to open a new savings account")
for ent in doc.ents:
    print(ent.text,ent.start_char,ent.end_char,ent.label_)
displacy.render(nlp(doc.text),style='ent',jupyter=True)

savings account 34 49 PRODUCT


In [15]:
doc = nlp("My credit card payment will be delayed")
for ent in doc.ents:
    print(ent.text,ent.start_char,ent.end_char,ent.label_)
displacy.render(nlp(doc.text),style='ent',jupyter=True)

credit card 3 14 PRODUCT


In [16]:
doc = nlp("What are the charges on credit card late payment in Bank of America")
for ent in doc.ents:
    print(ent.text,ent.start_char,ent.end_char,ent.label_)
displacy.render(nlp(doc.text),style='ent',jupyter=True)

credit card 24 35 PRODUCT
payment 41 48 ACTIVITY


In [17]:
doc = nlp("I lost my investment account password and cannot open my account now")
for ent in doc.ents:
    print(ent.text,ent.start_char,ent.end_char,ent.label_)
displacy.render(nlp(doc.text),style='ent',jupyter=True)

investment account 10 28 PRODUCT
account 57 64 PRODUCT


In [18]:
doc = nlp("What is the status of my loan account")
for ent in doc.ents:
    print(ent.text,ent.start_char,ent.end_char,ent.label_)
displacy.render(nlp(doc.text),style='ent',jupyter=True)

loan account 25 37 PRODUCT


In [19]:
doc = nlp("Australia wants to force Facebook and Google to pay media companies for news")
for ent in doc.ents:
    print(ent.text,ent.start_char,ent.end_char,ent.label_)
displacy.render(nlp(doc.text),style='ent',jupyter=True)

Australia 0 9 ACTIVITY
